# Introduction — LangGraph & Anthropic Claude

---

## What is LangChain?

**LangChain** is a framework for building applications powered by LLMs. It provides:
- Abstractions for prompts, models, output parsers
- Pre-built chains (e.g., `LLMChain`, `RetrievalQA`)
- Integrations with 100+ LLM providers, vector stores, and tools

LangChain chains are **linear pipelines** — they execute steps in a fixed sequence from start to finish. This works well for simple Q&A or RAG flows, but falls short when you need **loops**, **branching**, or **persistent state**.

---

## Why LangGraph?

LangGraph was built on top of LangChain specifically to handle **complex, stateful, multi-step workflows** that simple chains cannot model.

| Limitation in LangChain | How LangGraph solves it |
|---|---|
| No loops / cycles | Supports directed graphs with back-edges (cycles) |
| No shared state across steps | Every node reads/writes a single typed `StateDict` |
| No conditional branching | `add_conditional_edges` routes dynamically based on state |
| No pause-and-resume | `interrupt_before` + checkpointer enables Human-in-the-Loop |
| No persistent memory across calls | `MemorySaver` / `SqliteSaver` checkpointers persist state by `thread_id` |

---

## LangChain vs LangGraph — Key Differences

```
LangChain (chains)          LangGraph (graphs)
──────────────────          ──────────────────
A → B → C → END             START → [A] → [B] ──→ [C] → END
                                          └──→ [D] → END   ← conditional
                                          ↑__________|      ← cycle possible
```

| Feature | LangChain | LangGraph |
|---|---|---|
| **Execution model** | Linear pipeline | Directed graph (DAG or cyclic) |
| **State management** | Passed manually between steps | Shared `TypedDict` state — all nodes read/write |
| **Branching** | Limited (RunnableBranch) | Native conditional edges |
| **Loops / retries** | Not supported | Native cycles via back-edges |
| **Memory across turns** | Manual | Built-in checkpointers (`MemorySaver`, `SqliteSaver`) |
| **Human-in-the-Loop** | Not supported | `interrupt_before` / `interrupt_after` |
| **Observability** | LangSmith (basic) | LangSmith (full node-level traces) |

---

## When to use LangGraph

✅ **Use LangGraph when your workflow needs:**
- An **agent loop** — LLM decides whether to call tools, then loops back
- **Conditional routing** — different paths based on input or output
- **Multi-turn memory** — remember context across conversation turns
- **Human approval** — pause before a critical action
- **Retry / refinement loops** — keep improving output until quality threshold is met

❌ **Stick with LangChain chains when:**
- The workflow is a simple linear pipeline (A → B → C)
- No looping, branching, or state persistence is needed

---

## Core LangGraph Concepts

```
┌──────────────────────────────────────────────────────┐
│                  StateGraph                          │
│                                                      │
│   START  ──▶  [Node A]  ──▶  [Node B]  ──▶  END    │
│                  │                ▲                  │
│                  └────────────────┘  (cycle)        │
└──────────────────────────────────────────────────────┘
```

| Concept | Description |
|---|---|
| **`StateGraph`** | The graph container; takes a `TypedDict` schema defining the shared state |
| **State (`TypedDict`)** | Single source of truth — all nodes read from and write to it |
| **Node** | A Python function `(state) → dict` that returns only the keys it changes |
| **Edge** | `add_edge(A, B)` — fixed transition from node A to node B |
| **Conditional Edge** | `add_conditional_edges(A, fn)` — routing function decides the next node |
| **`START` / `END`** | Sentinel constants marking the graph entry and exit points |
| **Checkpointer** | Persists state between `.invoke()` calls (enables memory & HITL) |
| **`interrupt_before`** | Pauses execution before a node — waits for human input to resume |

# LangGraph + Anthropic Claude — Stateful AI Workflows

Build stateful AI workflows using **LangGraph** with **Anthropic Claude** (`claude-opus-4-6`) via `langchain-anthropic`.

| Component | Details |
|---|---|
| **LLM Provider** | Anthropic Claude |
| **Package** | `langchain-anthropic` |
| **Model Class** | `ChatAnthropic` |
| **Model** | `claude-opus-4-6` |
| **API Key** | `ANTHROPIC_API_KEY` (in `.env`) |
| **Graph Framework** | LangGraph (`StateGraph`, `START`, `END`) |

### Contents
- **Section 1** – Install & Setup
- **Part A** – StateGraph basics (nodes, fixed edges, START/END)
- **Part B** – LLM-powered pipeline (Summarise → Translate with Claude)
- **Part C** – Conditional routing (Math vs General Knowledge)

---

## Section 1 – Install & Setup

Install the required packages:
- **`langgraph`** – Core library (`StateGraph`, checkpointers, `START`/`END`)
- **`langchain-anthropic`** – Claude chat model integration via LangChain
- **`python-dotenv`** – Load `ANTHROPIC_API_KEY` from `.env`

In [ ]:
%pip install -q langgraph langchain-anthropic python-dotenv

In [ ]:
#(optional ,if you have an anthropic api key)
import os
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

# ── Claude via LangChain ──────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage,
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Load ANTHROPIC_API_KEY from .env
load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ── Initialise Claude chat model ──────────────────────────────────────────
chat_model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
)

from importlib.metadata import version
print("✅  Setup complete!")
print(f"   LangGraph       : {version('langgraph')}")
print(f"   langchain-anthropic : {version('langchain-anthropic')}")
print(f"   Model           : claude-haiku-4-5")

# Quick sanity check — confirm Claude responds
_test = chat_model.invoke([HumanMessage(content="Say hello in one word.")])
print(f"   Test call       : {_test.content.strip()}")

In [ ]:
#with llm gateway key
# STEP 1: Load .env FIRST
import os
from dotenv import load_dotenv
load_dotenv(override=True)



# STEP 3: Read LLM gateway vars (instead of ANTHROPIC_API_KEY from .env)
llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Push Anthropic-compatible env vars for LangChain internals
os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

# STEP 4: imports
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langsmith import Client, traceable

# STEP 5: status
print("=" * 55)
print("  ENVIRONMENT STATUS")
print("=" * 55)
print(f"  LLMGW_API_KEY         : {'✅ set (' + llmgw_api_key[:12] + '...)' if llmgw_api_key else '❌ MISSING'}")
print(f"  LLMGW_BASE_URL        : {llmgw_base_url!r}")
print("=" * 55)

# STEP 6: model + client
chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)

llm=chat_model
# Quick test
try:
    test_response = llm.invoke("Capital of France?")
    print(f"\nConnection Test: {test_response.content[:200]}...")
except Exception as e:
    print(f"Error: {e}")


---

## Part A – LangGraph Intro: StateGraph, Nodes & Edges

LangGraph models workflows as **directed graphs**:

```
START  →  [node_1]  →  [node_2]  →  END
```

Three core building blocks:

| Concept | Description |
|---------|-------------|
| **State** | A `TypedDict` — the single source of truth. Every node reads from it and returns a partial update (only the keys it changes). |
| **Node** | A Python function `(state: StateType) → dict`. Receives the full state; returns only the keys it wants to change. |
| **Edge** | A directed connection. `add_edge` is fixed; `add_conditional_edges` is dynamic. |

`START` and `END` are sentinel constants — `START` is the entry point; `END` signals the graph to stop.

> **Note:** This section contains no LLM calls — it demonstrates pure graph structure.

In [ ]:
# ── Basic StateGraph: nodes, fixed edges, START / END ────────────────────
print("PART A - LangGraph: StateGraph Basics")
print("=" * 55)

# ── 1. State schema ───────────────────────────────────────────────────────
class PipelineState(TypedDict):
    text:       str   # provided at invocation time
    char_count: int   # populated by node 1
    summary:    str   # populated by node 2

# ── 2. Node functions ─────────────────────────────────────────────────────
def count_chars(state: PipelineState) -> dict:
    """Node 1: count characters in the input text."""
    return {"char_count": len(state["text"])}

def make_summary(state: PipelineState) -> dict:
    """Node 2: build a human-readable summary string."""
    s = f"'{state['text']}' — {state['char_count']} characters."
    return {"summary": s}

# ── 3. Build the graph ────────────────────────────────────────────────────
builder_basic = StateGraph(PipelineState)
builder_basic.add_node("count",     count_chars)
builder_basic.add_node("summarise", make_summary)

# ── 4. Wire edges: START → count → summarise → END ───────────────────────
builder_basic.add_edge(START,       "count")
builder_basic.add_edge("count",     "summarise")
builder_basic.add_edge("summarise", END)

# ── 5. Compile and run ────────────────────────────────────────────────────
graph_basic = builder_basic.compile()
result = graph_basic.invoke({
    "text": "LangGraph makes stateful AI workflows easy!",
    "char_count": 0,
    "summary": "",
})

print("Input      :", result["text"])
print("Char count :", result["char_count"])
print("Summary    :", result["summary"])
print()
print("Node execution order: START → count → summarise → END")

---

##  – LLM-Powered Nodes (Claude)

Nodes can contain **any Python code** — including Claude API calls. Here we build a two-step NLP pipeline:

```
START  →  [summarise_node]  →  [translate_node]  →  END
```

Each node calls **Claude** (`claude-opus-4-6`) and updates a different field of the shared state.

- `summarise_node` — condenses the article to 2 sentences
- `translate_node` — translates the summary into French

In [ ]:
# ── LLM Pipeline: Summarise → Translate (Claude) ─────────────────────────
print("── LLM Pipeline: Summarise → Translate to French ──\n")

class NLPState(TypedDict):
    article:     str
    summary:     str
    translation: str

def summarise_node(state: NLPState) -> dict:
    """Node 1: Ask Claude to summarise the article to 2 sentences."""
    prompt = f"Summarise the following text in exactly 2 sentences:\n\n{state['article']}"
    resp = chat_model.invoke([HumanMessage(content=prompt)])
    return {"summary": resp.content.strip()}

def translate_node(state: NLPState) -> dict:
    """Node 2: Ask Claude to translate the summary to French."""
    prompt = f"Translate this text to French (keep it natural):\n\n{state['summary']}"
    resp = chat_model.invoke([HumanMessage(content=prompt)])
    return {"translation": resp.content.strip()}

# ── Build graph ───────────────────────────────────────────────────────────
builder_nlp = StateGraph(NLPState)
builder_nlp.add_node("summarise", summarise_node)
builder_nlp.add_node("translate", translate_node)

builder_nlp.add_edge(START,       "summarise")
builder_nlp.add_edge("summarise", "translate")
builder_nlp.add_edge("translate",  END)

nlp_graph = builder_nlp.compile()

# ── Run ───────────────────────────────────────────────────────────────────
article = (
    "Artificial intelligence is transforming industries worldwide. "
    "From healthcare diagnostics to autonomous vehicles, AI systems are "
    "performing tasks that previously required human expertise. "
    "The rapid adoption of large language models has further accelerated this trend."
)
r = nlp_graph.invoke({"article": article, "summary": "", "translation": ""})

print("Original Article:\n", article, "\n")
print("Summary (Claude):\n", r["summary"], "\n")
print("Translation — French (Claude):\n", r["translation"], "\n")

---

##  Conditional Routing with `add_conditional_edges`

A **routing function** inspects the current state and returns the **name of the next node** as a string. `add_conditional_edges(source_node, routing_fn)` wires it up dynamically.

```
START  →  [classify]  ──  route_question()  ──→  "math_node"    →  END
                                             └→  "general_node" →  END
```

- `classify_node` — keyword-based classification (no LLM needed, deterministic)
- `math_node` — Claude answers with math-assistant persona
- `general_node` — Claude answers with general-knowledge persona
- `route_question` — routing function; returns `"math_node"` or `"general_node"`

In [ ]:
# ── Conditional Routing: Math vs General Knowledge (Claude) ──────────────
print("── Conditional Routing: Math vs General Knowledge ──\n")

class RouterState(TypedDict):
    question: str
    category: str   # set by classify_node
    answer:   str   # set by the specialist node

MATH_KEYWORDS = {
    "calculate", "compute", "plus", "minus", "multiply", "times",
    "divide", "square root", "percent", "sum", "how much is",
}

def classify_node(state: RouterState) -> dict:
    """Classify the question as 'math' or 'general' (keyword-based, no LLM)."""
    q = state["question"].lower()
    cat = "math" if any(kw in q for kw in MATH_KEYWORDS) else "general"
    return {"category": cat}

def route_question(state: RouterState) -> Literal["math_node", "general_node"]:
    """Routing function: return the next node name based on category."""
    return f"{state['category']}_node"

def math_node(state: RouterState) -> dict:
    """Claude answers as a concise mathematics assistant."""
    resp = chat_model.invoke([
        SystemMessage(content="You are a concise mathematics assistant."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": resp.content.strip()}

def general_node(state: RouterState) -> dict:
    """Claude answers as a concise general-knowledge assistant."""
    resp = chat_model.invoke([
        SystemMessage(content="You are a concise general-knowledge assistant."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": resp.content.strip()}

# ── Build graph ───────────────────────────────────────────────────────────
builder_router = StateGraph(RouterState)
builder_router.add_node("classify",     classify_node)
builder_router.add_node("math_node",    math_node)
builder_router.add_node("general_node", general_node)

builder_router.add_edge(START, "classify")
builder_router.add_conditional_edges("classify", route_question)  # ← routing
builder_router.add_edge("math_node",    END)
builder_router.add_edge("general_node", END)

router_graph = builder_router.compile()

# ── Run with sample questions ─────────────────────────────────────────────
questions = [
    "What is the square root of 144?",
    "Who wrote the novel 1984?",
    "Calculate 256 divided by 8",
    "What is the capital of Australia?",
]
for q in questions:
    r = router_graph.invoke({"question": q, "category": "", "answer": ""})
    print(f"❓ {q}")
    print(f"   Category : {r['category']}")
    print(f"   Answer   : {r['answer'][:150].strip()}")
    print()

---

## Summary — LangGraph + Claude Conversion Reference

| Component | Google Gemini (Original) | Anthropic Claude (This Notebook) |
|---|---|---|
| **Package** | `langchain-google-genai` | `langchain-anthropic` |
| **Class** | `ChatGoogleGenerativeAI` | `ChatAnthropic` |
| **Model** | `gemini-2.5-flash` | `claude-opus-4-6` |
| **API Key env var** | `GEMINI_API_KEY` | `ANTHROPIC_API_KEY` |
| **Init param** | `google_api_key=...` | `anthropic_api_key=...` |
| **LangGraph** | ✅ unchanged | ✅ unchanged |
| **StateGraph / nodes / edges** | ✅ unchanged | ✅ unchanged |
| **Message types** | `HumanMessage`, `SystemMessage`, … | `HumanMessage`, `SystemMessage`, … |

### Key LangGraph Concepts Covered

- **`StateGraph`** — typed state machine with a `TypedDict` schema
- **Fixed edges** (`add_edge`) — deterministic node sequencing
- **Conditional edges** (`add_conditional_edges`) — dynamic routing based on state
- **`START` / `END`** — graph entry and exit sentinels
- **LLM nodes** — any node can call Claude (or any model) and return updated state fields